In [2]:
import sys
from pathlib import Path

# Get the project root (one directory above notebooks)
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python's import search path
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Source folder exists:", (PROJECT_ROOT / "src").exists())

Project root: c:\Users\KUSHAGRA\Desktop\SentinelAI
Project root exists: True
Source folder exists: True


In [3]:
from src.data_preprocessing import (
    load_data,
    remove_duplicates,
    separate_features_target,
    engineer_features
)

df_test = load_data(str(PROJECT_ROOT / "data/raw/creditcard.csv"))

df_test_clean = remove_duplicates(df_test)

X_test, y_test = separate_features_target(df_test_clean)

X_test_engineered = engineer_features(X_test)

print("\nFinal verification:")
print("Clean dataset:", df_test_clean.shape)
print("Features:", X_test.shape)
print("Engineered features:", X_test_engineered.shape)
print("Target:", y_test.shape)

Dataset loaded successfully!
Original dataset shape: (284807, 31)
Duplicate removal completed!
Original rows: 284807
Cleaned rows: 283726
Remaining duplicates: 0
Features shape: (283726, 30)
Target shape: (283726,)
Feature engineering completed!
Engineered features shape: (283726, 31)

Final verification:
Clean dataset: (283726, 31)
Features: (283726, 30)
Engineered features: (283726, 31)
Target: (283726,)


In [4]:
from pathlib import Path

print("PROJECT-LEVEL MODELS:")
project_models = PROJECT_ROOT / "models"

for file in project_models.glob("*"):
    print(file.name)

print("\nNOTEBOOK-LEVEL MODELS:")
notebook_models = PROJECT_ROOT / "notebooks" / "models"

for file in notebook_models.glob("*"):
    print(file.name)

PROJECT-LEVEL MODELS:

NOTEBOOK-LEVEL MODELS:
logistic_regression_baseline.pkl
logistic_regression_scaler.pkl
logistic_regression_threshold.pkl
random_forest_baseline.pkl
random_forest_scaler.pkl
random_forest_threshold.pkl
xgboost_baseline.pkl
xgboost_scaler.pkl
xgboost_threshold.pkl


In [5]:
print("Project-level model files:")

for file in (PROJECT_ROOT / "models").glob("*.pkl"):
    print(file.name)

print("\nTotal model files:",
      len(list((PROJECT_ROOT / "models").glob("*.pkl"))))


Project-level model files:
logistic_regression_baseline.pkl
logistic_regression_scaler.pkl
logistic_regression_threshold.pkl
random_forest_baseline.pkl
random_forest_scaler.pkl
random_forest_threshold.pkl
xgboost_baseline.pkl
xgboost_scaler.pkl
xgboost_threshold.pkl

Total model files: 9


In [ ]:
from src.model_loader import load_model_artifacts

rf_artifacts = load_model_artifacts("random_forest")

rf_model = rf_artifacts["model"]
rf_scaler = rf_artifacts["scaler"]
rf_threshold = rf_artifacts["threshold"]

print("\nModel verification:")
print("Model type:", type(rf_model).__name__)
print("Scaler type:", type(rf_scaler).__name__)
print("Classification threshold:", rf_threshold)                        

random_forest artifacts loaded successfully!
Model: random_forest_baseline.pkl
Scaler: random_forest_scaler.pkl
Threshold: 0.4

Model verification:
Model type: RandomForestClassifier
Scaler type: StandardScaler
Classification threshold: 0.4


In [7]:
from src.prediction_engine import predict_transaction

# Use a few unseen test transactions
sample_transactions = X_test.head(5).copy()

# Generate predictions
sample_results = predict_transaction(
    transaction_data=sample_transactions,
    model=rf_model,
    scaler=rf_scaler,
    threshold=rf_threshold
)

display(sample_results)

,Fraud Probability,Predicted Class,Prediction Label
0,0.0,0,Legitimate
1,0.0,0,Legitimate
2,0.0,0,Legitimate
3,0.0,0,Legitimate
4,0.0,0,Legitimate


In [12]:
for i, probability in enumerate(sample_results["Fraud Probability"]):
    print(f"Transaction {i}: {probability:.15e}")

print("\nUnique probabilities:")
print(sample_results["Fraud Probability"].unique())

Transaction 0: 0.000000000000000e+00
Transaction 1: 0.000000000000000e+00
Transaction 2: 0.000000000000000e+00
Transaction 3: 0.000000000000000e+00
Transaction 4: 0.000000000000000e+00

Unique probabilities:
[0.]


In [13]:
# Select five known fraud transactions from the test dataset
fraud_samples = X_test.loc[y_test == 1].head(5).copy()

# Generate predictions
fraud_results = predict_transaction(
    transaction_data=fraud_samples,
    model=rf_model,
    scaler=rf_scaler,
    threshold=rf_threshold
)

display(fraud_results)

,Fraud Probability,Predicted Class,Prediction Label
541,0.61,1,Fraud
623,0.64,1,Fraud
4920,0.39,0,Legitimate
6108,0.95,1,Fraud
6329,0.77,1,Fraud


In [17]:
from src.data_preprocessing import load_data, remove_duplicates

# Load and clean the original dataset
df_monitor = load_data(str(PROJECT_ROOT / "data/raw/creditcard.csv"))
df_monitor_clean = remove_duplicates(df_monitor)

# Recreate the same stratified train/test split
from sklearn.model_selection import train_test_split

X_monitor = df_monitor_clean.drop(columns=["Class"])
y_monitor = df_monitor_clean["Class"]

X_train_monitor, X_test_monitor, y_train_monitor, y_test_monitor = train_test_split(
    X_monitor,
    y_monitor,
    test_size=0.20,
    random_state=42,
    stratify=y_monitor
)

# Recreate the reference and current datasets
X_train_monitor, X_reference_train, y_train_monitor, y_reference_train = train_test_split(
    X_train_monitor,
    y_train_monitor,
    test_size=0.20,
    random_state=42,
    stratify=y_train_monitor
)

reference_data = X_reference_train.copy()
current_data = X_test_monitor.copy()

print("Reference data shape:", reference_data.shape)
print("Current data shape:", current_data.shape)

Dataset loaded successfully!
Original dataset shape: (284807, 31)
Duplicate removal completed!
Original rows: 284807
Cleaned rows: 283726
Remaining duplicates: 0
Reference data shape: (45396, 30)
Current data shape: (56746, 30)


In [18]:
reference_data = X_train_monitor.copy()
current_data = X_test_monitor.copy()

print("Reference data shape:", reference_data.shape)
print("Current data shape:", current_data.shape)

Reference data shape: (181584, 30)
Current data shape: (56746, 30)


In [19]:
feature_drift_test = detect_feature_drift(
    reference_data=reference_data,
    current_data=current_data
)

display(feature_drift_test.head(10))

print("Features tested:", len(feature_drift_test))
print("Initial drift signals:", feature_drift_test["Drift Detected"].sum())
print("Confirmed drift after correction:",
      feature_drift_test["Significant After Correction"].sum())

,Feature,KS Statistic,P-Value,Adjusted P-Value,Drift Detected,Significant After Correction
0,Time,0.0062283626,0.0695682924,0.5032625214,False,False
1,V1,0.0042793992,0.4058152793,0.9049636914,False,False
2,V2,0.0029952220,0.8315834298,0.9361506441,False,False
3,V3,0.0043679641,0.3803805249,0.9049636914,False,False
4,V4,0.0033318808,0.7218639688,0.9361506441,False,False
5,V5,0.0048119469,0.2685546914,0.8951823046,False,False
6,V6,0.0033462042,0.7169261917,0.9361506441,False,False
7,V7,0.0036399405,0.6142979154,0.9361506441,False,False
8,V8,0.0035717460,0.6381617300,0.9361506441,False,False
9,V9,0.0031737461,0.7752288717,0.9361506441,False,False


Features tested: 30
Initial drift signals: 3
Confirmed drift after correction: 0


In [23]:
import importlib
import src.drift_monitor as drift_monitor

# Reload the latest saved Python file
importlib.reload(drift_monitor)

# Import the functions
detect_feature_drift = drift_monitor.detect_feature_drift
detect_prediction_drift = drift_monitor.detect_prediction_drift

print("Both drift monitoring functions imported successfully!")

Both drift monitoring functions imported successfully!


In [25]:
# Load Random Forest artifacts
rf_artifacts = load_model_artifacts("random_forest")

rf_model = rf_artifacts["model"]
rf_scaler = rf_artifacts["scaler"]

# Scale reference and current datasets
reference_scaled = pd.DataFrame(
    rf_scaler.transform(reference_data),
    columns=reference_data.columns
)

current_scaled = pd.DataFrame(
    rf_scaler.transform(current_data),
    columns=current_data.columns
)

# Generate fraud probabilities
reference_probabilities = rf_model.predict_proba(reference_scaled)[:, 1]
current_probabilities = rf_model.predict_proba(current_scaled)[:, 1]

# Detect prediction drift
prediction_drift_test = detect_prediction_drift(
    reference_probabilities=reference_probabilities,
    current_probabilities=current_probabilities
)

display(prediction_drift_test)

random_forest artifacts loaded successfully!
Model: random_forest_baseline.pkl
Scaler: random_forest_scaler.pkl
Threshold: 0.4


{'KS Statistic': np.float64(0.011703256326838551),
 'P-Value': np.float64(1.4254712525359086e-05),
 'Drift Detected': True,
 'Reference Mean': np.float64(0.0016658956736276323),
 'Current Mean': np.float64(0.0015495365312092483)}

In [26]:
import importlib
import src.drift_monitor as drift_monitor

# Reload the updated module
importlib.reload(drift_monitor)

# Generate combined report
monitoring_report = drift_monitor.generate_monitoring_report(
    feature_drift_df=feature_drift_test,
    prediction_drift_result=prediction_drift_test
)

display(monitoring_report)

,Overall Status,Total Features Monitored,Initial Feature Drift Signals,Confirmed Feature Drift,Prediction Drift Detected,Prediction KS Statistic,Prediction P-Value,Reference Mean Probability,Current Mean Probability
0,DRIFT DETECTED,30,3,0,True,0.0117032563,0.0000142547,0.0016658957,0.0015495365


In [27]:
from src.model_evaluation import evaluate_model

# Evaluate Random Forest on the test dataset
evaluation_results = evaluate_model(
    y_true=y_test_monitor,
    y_probabilities=current_probabilities,
    threshold=rf_artifacts["threshold"]
)

print("Model Evaluation Metrics:")
display(evaluation_results["metrics"])

print("Confusion Matrix:")
display(evaluation_results["confusion_matrix"])

Model Evaluation Metrics:


,Threshold,Accuracy,Precision,Recall,F1 Score,PR-AUC (Average Precision)
0,0.4,0.9995065732,0.9466666667,0.7473684211,0.8352941176,0.7770005894


Confusion Matrix:


,Predicted Legitimate,Predicted Fraud
Actual Legitimate,56647,4
Actual Fraud,24,71


In [32]:
from pathlib import Path
import json

# Save the complete JSON request
payload_file = PROJECT_ROOT / "api_request.json"

payload_file.write_text(
    json.dumps(sample_payload, indent=2),
    encoding="utf-8"
)

print("JSON saved successfully!")
print("File:", payload_file)
print("Number of features:", len(sample_payload["features"]))

JSON saved successfully!
File: c:\Users\KUSHAGRA\Desktop\SentinelAI\api_request.json
Number of features: 30


In [33]:
import json

# Select two fraud transactions
fraud_samples = X_test_monitor.loc[
    y_test_monitor == 1
].head(2)

# Select one legitimate transaction
legitimate_sample = X_test_monitor.loc[
    y_test_monitor == 0
].head(1)

# Combine all three
batch_samples = pd.concat(
    [fraud_samples, legitimate_sample]
)

# Prepare API payload
batch_payload = {
    "transactions": [
        {"features": row.astype(float).to_dict()}
        for _, row in batch_samples.iterrows()
    ]
}

# Save complete JSON
batch_file = PROJECT_ROOT / "batch_api_request.json"

batch_file.write_text(
    json.dumps(batch_payload, indent=2),
    encoding="utf-8"
)

print("Batch payload created successfully!")
print("Transactions:", len(batch_payload["transactions"]))
print("File:", batch_file)

Batch payload created successfully!
Transactions: 3
File: c:\Users\KUSHAGRA\Desktop\SentinelAI\batch_api_request.json


In [36]:
import importlib
import src.api as api

# Reload the latest API module
importlib.reload(api)

# Import the validation function
validate_transaction_features = api.validate_transaction_features

print("Validation function imported successfully!")

random_forest artifacts loaded successfully!
Model: random_forest_baseline.pkl
Scaler: random_forest_scaler.pkl
Threshold: 0.4
random_forest artifacts loaded successfully!
Model: random_forest_baseline.pkl
Scaler: random_forest_scaler.pkl
Threshold: 0.4
Validation function imported successfully!


In [37]:
import numpy as np
from fastapi import HTTPException

# Create a valid transaction
test_features = X_test_monitor.iloc[0].astype(float).to_dict()

# Test NaN
nan_features = test_features.copy()
nan_features["Amount"] = np.nan

try:
    validate_transaction_features(nan_features)
except HTTPException as error:
    print("NaN test:", error.status_code, error.detail)

# Test Infinity
inf_features = test_features.copy()
inf_features["Amount"] = np.inf

try:
    validate_transaction_features(inf_features)
except HTTPException as error:
    print("Infinity test:", error.status_code, error.detail)

NaN test: 422 {'message': 'Transaction contains invalid numerical values', 'invalid_features': ['Amount']}
Infinity test: 422 {'message': 'Transaction contains invalid numerical values', 'invalid_features': ['Amount']}


In [38]:
# Create a copy of the valid batch payload
invalid_batch_payload = {
    "transactions": [
        dict(transaction)
        for transaction in batch_payload["transactions"]
    ]
}

# Add an unexpected feature to transaction 1
invalid_batch_payload["transactions"][1]["features"]["UnknownFeature"] = 123.45

# Save the test payload
invalid_batch_file = PROJECT_ROOT / "invalid_batch_request.json"

invalid_batch_file.write_text(
    json.dumps(invalid_batch_payload, indent=2),
    encoding="utf-8"
)

print("Invalid batch payload created!")

Invalid batch payload created!


In [39]:
import json

# Select one legitimate transaction
legitimate_sample = X_test_monitor.loc[
    y_test_monitor == 0
].iloc[0].astype(float).to_dict()

# Select one fraudulent transaction
fraud_sample = X_test_monitor.loc[
    y_test_monitor == 1
].iloc[0].astype(float).to_dict()

# Create sample transaction collection
dashboard_samples = {
    "Legitimate": legitimate_sample,
    "Fraudulent": fraud_sample
}

# Save samples in the processed data folder
sample_file = PROJECT_ROOT / "data" / "processed" / "dashboard_samples.json"

sample_file.write_text(
    json.dumps(dashboard_samples, indent=2),
    encoding="utf-8"
)

print("Dashboard samples saved successfully!")
print("Legitimate features:", len(legitimate_sample))
print("Fraudulent features:", len(fraud_sample))
print("File:", sample_file)

Dashboard samples saved successfully!
Legitimate features: 30
Fraudulent features: 30
File: c:\Users\KUSHAGRA\Desktop\SentinelAI\data\processed\dashboard_samples.json


In [40]:
batch_test_file = PROJECT_ROOT / "data" / "processed" / "batch_test.csv"

X_test_monitor.head(100).to_csv(
    batch_test_file,
    index=False
)

print("Batch test file created:", batch_test_file)
print("Shape:", X_test_monitor.head(100).shape)

Batch test file created: c:\Users\KUSHAGRA\Desktop\SentinelAI\data\processed\batch_test.csv
Shape: (100, 30)


In [41]:
# Select 50 legitimate transactions
legitimate_batch = X_test_monitor.loc[
    y_test_monitor == 0
].head(50)

# Select 50 fraudulent transactions
fraudulent_batch = X_test_monitor.loc[
    y_test_monitor == 1
].head(50)

# Combine and shuffle
mixed_batch = pd.concat(
    [legitimate_batch, fraudulent_batch]
).sample(frac=1, random_state=42)

# Save CSV
mixed_batch_file = (
    PROJECT_ROOT / "data" / "processed" / "mixed_batch_test.csv"
)

mixed_batch.to_csv(mixed_batch_file, index=False)

print("Mixed batch shape:", mixed_batch.shape)
print("Known legitimate:", len(legitimate_batch))
print("Known fraudulent:", len(fraudulent_batch))

Mixed batch shape: (100, 30)
Known legitimate: 50
Known fraudulent: 50
